# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
df = pd.read_csv('data/AviationData.csv', encoding='latin-1', low_memory=False)

print("Shape:", df.shape)
print("\n--- Data Types & Non-Null Counts ---")
df.info()

print("\n--- Summary Statistics ---")
display(df.describe())

print("\n--- Missing Values ---")
display(df.isna().sum())

print("\n--- Column Names ---")
print(df.columns.tolist())

Shape: (88889, 31)

--- Data Types & Non-Null Counts ---
<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str  

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000



--- Missing Values ---


Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i


--- Column Names ---
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date']


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

#### Aircraft.Category

`value_counts(dropna=False)` reveals that NaN entries make up a significant portion of the data. Because the column was introduced later in the NTSB database, most older records have NaN — but those older records describe conventional airplanes. Dropping all NaN values here would eliminate a large share of usable data. We therefore **keep rows where `Aircraft.Category` is `'Airplane'` OR NaN**, and drop only the non-airplane categories (helicopter, glider, balloon, etc.).

In [3]:
# --- Aircraft.Category ---
print("Aircraft.Category value counts:")
print(df['Aircraft.Category'].value_counts(dropna=False))

# Keep airplanes and NaN (NaN likely means airplane for older records)
df = df[df['Aircraft.Category'].isin(['Airplane']) | df['Aircraft.Category'].isna()]
print(f"\nAfter Aircraft.Category filter: {df.shape[0]} rows")

# --- Amateur.Built ---
print("\nAmateur.Built value counts:")
print(df['Amateur.Built'].value_counts(dropna=False))

# Keep only professionally built aircraft
df = df[df['Amateur.Built'] == 'No']
print(f"After Amateur.Built filter: {df.shape[0]} rows")

# --- Event.Date ---
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
print(f"\nDate range before filter: {df['Event.Date'].min()} to {df['Event.Date'].max()}")

# Keep events from 1983 onwards (40-year active window for a plane retired by 2023)
df = df[df['Event.Date'] >= '1983-01-01']
print(f"After date filter (>=1983): {df.shape[0]} rows")
print(f"Date range after filter: {df['Event.Date'].min()} to {df['Event.Date'].max()}")

Aircraft.Category value counts:
Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

After Aircraft.Category filter: 84219 rows

Amateur.Built value counts:
Amateur.Built
No     76008
Yes     8111
NaN      100
Name: count, dtype: int64
After Amateur.Built filter: 76008 rows

Date range before filter: 1948-10-24 00:00:00 to 2022-12-29 00:00:00
After date filter (>=1983): 73002 rows
Date range after filter: 1983-01-01 00:00:00 to 2022-12-29 00:00:00


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries',
               'Total.Minor.Injuries', 'Total.Uninjured']

print("Missing values in injury columns:")
print(df[injury_cols].isna().sum())
print()

# Assumption: a missing injury count is treated as zero.
# When an accident report omits a count it most likely means no one fell
# into that category.  This is a standard imputation choice for this dataset.
df[injury_cols] = df[injury_cols].fillna(0)

# Total people on board = sum of all injury categories
df['Total.People.On.Board'] = df[injury_cols].sum(axis=1)

# Serious.Fatal.Fraction = (Fatal + Serious) / Total on board
# Only compute where total > 0 to avoid division by zero
df['Serious.Fatal.Fraction'] = np.where(
    df['Total.People.On.Board'] > 0,
    (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.People.On.Board'],
    np.nan
)

# Drop rows where total people on board is 0 (no useful injury data)
before = df.shape[0]
df = df[df['Total.People.On.Board'] > 0]
print(f"Dropped {before - df.shape[0]} rows with 0 people on board.")
print(f"Remaining rows: {df.shape[0]}")

print("\nSerious.Fatal.Fraction summary:")
display(df['Serious.Fatal.Fraction'].describe())

Missing values in injury columns:
Total.Fatal.Injuries       9634
Total.Serious.Injuries    10605
Total.Minor.Injuries      10114
Total.Uninjured            4714
dtype: int64

Dropped 1178 rows with 0 people on board.
Remaining rows: 71824

Serious.Fatal.Fraction summary:


count    71824.000000
mean         0.271188
std          0.427856
min          0.000000
25%          0.000000
50%          0.000000
75%          0.666667
max          1.000000
Name: Serious.Fatal.Fraction, dtype: float64

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
print("Aircraft.damage value counts (including NaN):")
print(df['Aircraft.damage'].value_counts(dropna=False))

# Standardize casing
df['Aircraft.damage'] = df['Aircraft.damage'].str.strip().str.title()

# Replace unknown / rare values with NaN
df['Aircraft.damage'] = df['Aircraft.damage'].replace({'Unknown': np.nan})

print("\nAfter cleaning:")
print(df['Aircraft.damage'].value_counts(dropna=False))

# Binary column: 1 if Destroyed, 0 otherwise (NaN stays NaN)
df['Destroyed'] = np.where(
    df['Aircraft.damage'].isna(), np.nan,
    np.where(df['Aircraft.damage'] == 'Destroyed', 1, 0)
)

print("\nDestroyed column distribution:")
print(df['Destroyed'].value_counts(dropna=False))

Aircraft.damage value counts (including NaN):
Aircraft.damage
Substantial    52328
Destroyed      14888
NaN             2348
Minor           2186
Unknown           74
Name: count, dtype: int64

After cleaning:
Aircraft.damage
Substantial    52328
Destroyed      14888
NaN             2422
Minor           2186
Name: count, dtype: int64

Destroyed column distribution:
Destroyed
0.0    54514
1.0    14888
NaN     2422
Name: count, dtype: int64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

**Observed problems with the `Make` column:**

1. **Inconsistent casing** — e.g. `CESSNA`, `cessna`, `Cessna` all refer to the same manufacturer.
2. **Leading/trailing whitespace** — some entries have extra spaces.
3. **Spelling variants of the same manufacturer** — e.g. `Mcdonnell Douglas`, `Mcdonnell-Douglas`; `De Havilland`, `Dehavilland`; `Air Tractor` vs `Air Tractor Inc`; `Aviat` vs `Aviat Aircraft Inc`; `Champion` vs `American Champion Aircraft` (its successor company).
4. **Abbreviations and legacy names** — some entries use a division name or abbreviation instead of the parent company.
5. **Non-airplane manufacturers** — because we kept rows where `Aircraft.Category` is NaN, some helicopter makers (`Bell`, `Robinson`, `Hiller`, `Enstrom`…) and balloon makers (`Balloon Works`, whose models are all *Firefly* hot-air balloons) slipped through.
6. **Glider models under mixed manufacturers** — a model-level check shows that `Let` records are almost all *L-13 / L-23 Blaník* gliders (only the *L-410* is an airplane), and `Burkhart Grob` records are *G102 / G103* gliders and *G109* motor-gliders.

**Cleaning plan:** strip whitespace → convert to title case → merge known variants with `replace()` → verify suspect makes at model level → remove non-airplane makes and glider models → keep only makes with ≥ 50 accidents.

In [6]:
# Basic cleaning: strip whitespace, title case
df['Make'] = df['Make'].str.strip().str.title()

# Merge common variants into canonical names
make_replacements = {
    'Mcdonnell Douglas': 'Mcdonnell Douglas',
    'Mcdonnell-Douglas': 'Mcdonnell Douglas',
    'Dehavilland': 'De Havilland',
    'De Havilland Canada': 'De Havilland',
    'Bombardier Inc': 'Bombardier',
    'Embraer-Empresa Brasileira De Aeronautica S.A.': 'Embraer',
    'Embraer S.A.': 'Embraer',
    'Empresa Brasileira De Aeronautica S.A. (Embraer)': 'Embraer',
    'Airbus Industrie': 'Airbus',
    'Raytheon Aircraft Company': 'Raytheon',
    'Raytheon Corporate Jets': 'Raytheon',
    'Cirrus Design': 'Cirrus',
    'Cirrus Design Corp': 'Cirrus',
    'Maule Air': 'Maule',
    'Maule Air Inc': 'Maule',
    'Air Tractor Inc': 'Air Tractor',
    'Aviat Aircraft Inc': 'Aviat',
    'American Champion Aircraft': 'Champion',
    'Grob Aircraft Ag': 'Burkhart Grob',
}
df['Make'] = df['Make'].replace(make_replacements)
print(f"Unique makes after merging variants: {df['Make'].nunique()}")

Unique makes after merging variants: 1516


**Verification of suspect makes.** Before removing anything, we inspect the models recorded under makes that are not obviously airplane manufacturers, and check that name variants have been merged. `\b` in the regular expression matches whole words only, so `Aviat` does not match `Aviation`.

In [7]:
# Check 1: most frequent models for each suspect make
suspects = ['Let', 'Burkhart Grob', 'Enstrom', 'Hiller', 'Balloon Works']
for make in suspects:
    print(f"\n=== {make} ===")
    print(df.loc[df['Make'] == make, 'Model'].value_counts().head(10).to_string())

# Check 2: remaining name variants (should show one name per manufacturer)
mask = df['Make'].str.contains(r'\bAviat\b|Air Tractor|Champion|Diamond|Grob',
                               case=False, na=False, regex=True)
print("\n=== Name variants ===")
print(df.loc[mask, 'Make'].value_counts().to_string())


=== Let ===
Model
L-13                 48
BLANIK L-13          21
L-23                  6
410 UVP-E             4
Blanik L-13           4
L-23 SUPER BLANIK     3
SUPER BLANIK L-23     2
L13                   1
LF-13                 1
SUPER BLANIK L-33     1

=== Burkhart Grob ===
Model
G103               15
G102                6
103                 6
G-103A              6
G-103               5
G-109B              5
G103 TWIN ASTIR     4
103A                3
G-102               3
G109B               3

=== Enstrom ===
Model
F-28C    43
280C     38
F-28A    36
F-28F    21
F28C     14
F28A     13
F28F      9
280FX     9
F28       3
480       3

=== Hiller ===
Model
UH-12E     132
UH-12B      19
UH-12C      19
UH-12D      16
FH-1100     13
UH12E       11
12E         10
UH-12ET      6
UH-12A       5
UH12-E       3

=== Balloon Works ===
Model
FIREFLY 7       36
FIREFLY 8-24     8
FIREFLY 8B       6
FIREFLY 7-B      5
FIREFLY 7-15     5
FIREFLY 7B       4
FIREFLY 8        4
FIREFLY-7      

**Removing non-airplanes.** Helicopter and balloon manufacturers are removed entirely. For `Let` and `Burkhart Grob` we filter at model level instead, so that genuine airplanes (the *L-410* transport, and *G115 / G120* trainers if present) are kept while gliders and motor-gliders are excluded. Finally, only makes with at least 50 accidents are kept, so that rates are computed on a sufficient sample.

In [8]:
# Remove helicopter and balloon manufacturers that slipped in via NaN Aircraft.Category
non_airplane_makes = ['Bell', 'Robinson', 'Hughes', 'Schweizer', 'Sikorsky',
                      'Eurocopter', 'Agusta', 'Enstrom', 'Hiller', 'Balloon Works']
before = df.shape[0]
df = df[~df['Make'].isin(non_airplane_makes)]
print(f"Removed {before - df.shape[0]} rows from non-airplane makes: {non_airplane_makes}")

# Remove gliders: Let (except L-410) and Grob (except G115/G120 trainers)
glider_mask = (
    (df['Make'] == 'Let') & ~df['Model'].str.contains('410', na=False)
) | (
    (df['Make'] == 'Burkhart Grob') & ~df['Model'].str.contains('115|120', na=False)
)
before = df.shape[0]
df = df[~glider_mask]
print(f"Removed {before - df.shape[0]} glider rows (Let, Burkhart Grob)")

print("\nTop 30 makes after cleaning:")
print(df['Make'].value_counts().head(30).to_string())

# Keep only makes with at least 50 accidents
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
before = df.shape[0]
df = df[df['Make'].isin(valid_makes)]
print(f"\nKept {len(valid_makes)} makes with ≥50 accidents.")
print(f"Rows dropped: {before - df.shape[0]}, remaining: {df.shape[0]}")

Removed 4532 rows from non-airplane makes: ['Bell', 'Robinson', 'Hughes', 'Schweizer', 'Sikorsky', 'Eurocopter', 'Agusta', 'Enstrom', 'Hiller', 'Balloon Works']
Removed 194 glider rows (Let, Burkhart Grob)

Top 30 makes after cleaning:
Make
Cessna                            25640
Piper                             14058
Beech                              5068
Boeing                             2077
Mooney                             1268
Grumman                            1064
Bellanca                            977
Air Tractor                         887
Aeronca                             606
Maule                               570
Champion                            558
Mcdonnell Douglas                   541
De Havilland                        499
Stinson                             421
Aero Commander                      410
Luscombe                            390
Cirrus                              373
North American                      369
Taylorcraft                         366

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [9]:
# Drop rows with no model info
print(f"Model NaN count: {df['Model'].isna().sum()}")
df = df.dropna(subset=['Model'])

# Clean model names: strip whitespace, uppercase to merge casing variants
df['Model'] = df['Model'].str.strip().str.upper()

# Check if model names are unique to each make
model_make_counts = df.groupby('Model')['Make'].nunique()
shared_models = model_make_counts[model_make_counts > 1]
print(f"\nModels shared across multiple makes: {len(shared_models)}")
display(shared_models.head(10))

# Since some model names appear under multiple makes,
# create a unique identifier combining make and model
df['Make_Model'] = df['Make'] + ' ' + df['Model']

print(f"\nUnique Make_Model combinations: {df['Make_Model'].nunique()}")
print(f"Remaining rows: {df.shape[0]}")

Model NaN count: 14

Models shared across multiple makes: 305


Model
100        4
100-180    2
112        4
112A       3
112TC      2
112TCA     3
114        2
114A       2
120        2
140        2
Name: Make, dtype: int64


Unique Make_Model combinations: 5289
Remaining rows: 61848


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [10]:
other_cols = ['Engine.Type', 'Weather.Condition', 'Number.of.Engines',
              'Purpose.of.flight', 'Broad.phase.of.flight']

for col in other_cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(df[col].value_counts(dropna=False))

# --- Engine.Type ---
df['Engine.Type'] = df['Engine.Type'].str.strip().str.title()
df['Engine.Type'] = df['Engine.Type'].replace({
    'Unknown': np.nan, 'Unk': np.nan
})
# Group rare engine types (fewer than 50 occurrences) into "Other"
engine_counts = df['Engine.Type'].value_counts()
rare_engines = engine_counts[engine_counts < 50].index
df['Engine.Type'] = df['Engine.Type'].replace({e: 'Other' for e in rare_engines})

# --- Weather.Condition ---
df['Weather.Condition'] = df['Weather.Condition'].str.strip().str.upper()
df['Weather.Condition'] = df['Weather.Condition'].replace({
    'UNK': np.nan, 'UNKNOWN': np.nan
})

# --- Number.of.Engines ---
print(f"\n\nNumber.of.Engines unique values before cleaning: {sorted(df['Number.of.Engines'].dropna().unique())}")
# 0 engines on an airplane is almost certainly a placeholder → treat as NaN
df['Number.of.Engines'] = df['Number.of.Engines'].replace({0: np.nan})
print(f"Number.of.Engines unique values after cleaning: {sorted(df['Number.of.Engines'].dropna().unique())}")

# --- Purpose.of.flight ---
df['Purpose.of.flight'] = df['Purpose.of.flight'].str.strip().str.title()
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({
    'Unknown': np.nan
})
# Group rare flight purposes (fewer than 50 occurrences) into "Other"
purpose_counts = df['Purpose.of.flight'].value_counts()
rare_purposes = purpose_counts[purpose_counts < 50].index
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({p: 'Other' for p in rare_purposes})

# --- Broad.phase.of.flight ---
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.strip().str.title()
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace({
    'Unknown': np.nan, 'Other': np.nan
})

print("\n\n--- Cleaned value counts ---")
for col in other_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(15))


Column: Engine.Type
Engine.Type
Reciprocating      51759
NaN                 3593
Turbo Prop          2746
Turbo Fan           2058
Unknown              774
Turbo Jet            528
Turbo Shaft          388
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64

Column: Weather.Condition
Weather.Condition
VMC    53827
IMC     4975
NaN     2266
UNK      605
Unk      175
Name: count, dtype: int64

Column: Number.of.Engines
Number.of.Engines
1.0    48733
2.0     9106
NaN     2963
3.0      425
4.0      379
0.0      242
Name: count, dtype: int64

Column: Purpose.of.flight
Purpose.of.flight
Personal                     35175
Instructional                 8081
Unknown                       4720
NaN                           3687
Aerial Application            3266
Business                      3008
Positioning                   1086
Other Work Use                 572
Ferry                          558
Aerial Observation             393
Executive/corporate            371
P

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [11]:
# Keep only columns with more than 20,000 non-null values
non_null_counts = df.notna().sum()
print("Non-null counts per column:")
print(non_null_counts.sort_values())

threshold = 20000
cols_to_keep = non_null_counts[non_null_counts > threshold].index.tolist()
cols_to_drop = [c for c in df.columns if c not in cols_to_keep]

print(f"\nColumns dropped (≤{threshold} non-null): {cols_to_drop}")
print(f"Columns kept: {len(cols_to_keep)}")

df = df[cols_to_keep]
print(f"\nFinal shape: {df.shape}")
df.info()

Non-null counts per column:
Schedule                   8738
Air.carrier               10921
FAR.Description           17830
Aircraft.Category         17910
Longitude                 22507
Latitude                  22512
Airport.Code              37230
Airport.Name              38941
Broad.phase.of.flight     44735
Publication.Date          51068
Purpose.of.flight         53441
Engine.Type               57480
Report.Status             58341
Number.of.Engines         58643
Weather.Condition         58802
Aircraft.damage           59842
Destroyed                 59842
Registration.Number       61101
Country                   61735
Location                  61817
Event.Date                61848
Accident.Number           61848
Investigation.Type        61848
Injury.Severity           61848
Model                     61848
Make                      61848
Total.Fatal.Injuries      61848
Amateur.Built             61848
Total.Uninjured           61848
Total.Minor.Injuries      61848
Total.Seriou

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [12]:
df.to_csv('data/AviationData_cleaned.csv', index=False)
print("Cleaned data saved to data/AviationData_cleaned.csv")
print(f"Final dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Cleaned data saved to data/AviationData_cleaned.csv
Final dataset: 61848 rows, 31 columns
